In [ ]:
%reload_ext autoreload
%autoreload 2
import numpy as np
import matplotlib.pylab as plt

from scipy.stats import multivariate_normal as norm
plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['image.cmap'] = 'Grays'

# Fix the random seed to facilitate grading
np.random.seed(1)

# HW3.2 - Global Optimization

**Preparation** For all functoins below where it says return None, copy the contents over from HW3.1:

In [ ]:
def phi(x, K):
    return np.power(np.atleast_1d(x)[:, None], np.arange(0, K))

def y_est(x, w):
    return phi(x, K=len(w)) @ w

In [ ]:
def analytical_posterior(m_prev, S_prev, x, t, beta):
    return None

In [ ]:
def predictive_distribution(x, mu, S, beta):
    return None

def plot_function_distribution(ax, x_grid, t_true, data, mu, S):
    K = len(mu)
    ax.plot(x_grid, t_true, "C0", label="Ground Truth")
    [ax.scatter(x, t, s=50, zorder=10) for x, t in data]
    ax.plot(x_grid, y_est(x_grid, mu), "C1", label="Mean")
    Phi = phi(x_grid, K)              
    means = Phi @ mu
    vars_ = np.sum((Phi @ S) * Phi, axis=1)  # (N,)
    stds = np.sqrt(vars_)
    ax.plot(x_grid, y_est(x_grid, mu) + stds, "C1", label="+- Sigma")
    ax.plot(x_grid, y_est(x_grid, mu) - stds, "C1", label="+- Sigma")

    # Sample weights from posterior
    post = norm(mu, S)
    w_samples = post.rvs(10)
    for w_k in w_samples:
        ax.plot(x_grid, y_est(x_grid, w_k), "k-", alpha=0.3)
    ax.set_title("Function Samples")
    ax.set_ylim(-1, 1)

def plot_predictive_distribution(ax, x_grid, beta, K, mu, S):
    t_grid = np.linspace(-1, 1, 200)
    predict_grid = np.empty((len(x_grid), len(t_grid)))
    for i, x in enumerate(x_grid):
        mean, variance = predictive_distribution(x, mu, S, beta)
        predict = norm(mean, np.sqrt(variance))  # takes in covariance
        predict_grid[i, :] = predict.pdf(t_grid)
        ax.scatter(x, mean, color="C0", s=5)
        ax.scatter(x, mean + np.sqrt(variance), color="C1", s=5)
        ax.scatter(x, mean - np.sqrt(variance), color="C1", s=5)
    ax.contourf(x_grid, t_grid, predict_grid.T, levels=30, zorder=-1)
    ax.set_title("Predictive Distribution")

# 2.a Bayesian Optimization

**Bayesian optimization** is a strategy for finding the minimum of an unknown function when evaluations are limited or expensive. Instead of testing many points blindly, we build a probabilistic model of the function from the observations collected so far, then use that model to decide where to sample next.

In this example, the blue curve is the hidden function we want to optimize, and the orange points are noisy measurements of that function. After each new observation, we update our belief about the function using Bayesian inference. We then compute an **acquisition function**, which balances two goals:

- **exploration**: sample where the model is still uncertain,
- **exploitation**: sample where the function is likely to have a low value.

By repeating this update-and-sample process, the algorithm gradually learns where the optimum is located.

### Data setup 

We begin by generating a **nonconvex function** and noisy measurement / function evaluations. 

In [ ]:
def y(x, w):
    return w[0] + w[1] * np.sin(2 * np.pi * x) + w[2] * x**2
    
def generate_data(w, beta, x=None):
    noise = np.random.normal(0, np.sqrt(beta))
    if x is None:
        x = np.random.rand() * 1.5 - 0.5
    t = y(x, w) + noise
    return np.array([x]), float(t.flatten()[0])

def plot_acquisition_functions_optimization(ax, x_grid, mu, S, beta, t_best, kwargs):
    j = 0
    for function in ALL_FUNCTIONS:
        acquisition_grid = [
            function(np.array([x]), mu, S, beta, t_best=t_best, **kwargs) for x in x_grid
        ]
        acquisition_grid = (acquisition_grid - np.min(acquisition_grid)) / (
            np.max(acquisition_grid) - np.min(acquisition_grid)
        )
        name = "".join([a[0].capitalize() for a in function.__name__.split("_")])
        ax.plot(x_grid, acquisition_grid, label=name, color=f"C{j}", ls=":")
        j += 1
    ax.legend(bbox_to_anchor=[0.0, 0.0], loc="lower left")
    
np.random.seed(1)

# ground truth parameters
w_true = np.array([-0.3, 0.5, 1.0])  # ground-truth parameters
beta = 0.05**2 # variance

# prior parameters
K = 5
m_0 = np.zeros(K)  # mean
alpha = 0.5
S_0 = alpha * np.eye(K)  # covariance

# plotting parameters
x_grid = np.linspace(-0.5, 1, 100)
t_true = [y(x, w_true) for x in x_grid]

plt.figure()
plt.plot(x_grid, t_true, color="C0")
for i in range(100):
    x, t = generate_data(w=w_true, beta=beta)
    plt.scatter(x, t, color="C1")
plt.title("Function to optimize")
plt.show()

### Acquisition Functions for Bayesian Optimization

In Bayesian optimization, the next query point is selected by maximizing an **acquisition function**. An acquisition function uses the predictive mean and uncertainty of the surrogate model to decide where to sample next.

(10 pts) **2.a.1** Your task is to implement the following three acquisition functions:

1. **Expected Improvement (EI)**  
   Measures the expected amount by which sampling at `x` improves over the current best observed value `y_best`.

2. **Probability of Improvement (PI)**  
   Measures the probability that sampling at `x` yields a value smaller than `y_best`.

3. **Lower Confidence Bound (LCB)**  
   Balances low predicted mean and high uncertainty. For minimization, points with smaller lower confidence bound are preferred.

Below, x is the point at which we evaluate the acquisition function, (mu, S) define the posterior distribution over weights, beta is the measurement noise standard deviation, and t_best is the best value found so far. The parameters epsilon and gamma trade off exploration and exploitation as explained in class. 

In [ ]:
### ============= 2.a.1 Fill in below ==============
### Implement the following acquisition functions for bayesian optimization:

def expected_improvement(x, mu, S, beta, t_best, epsilon, **kwargs):
    return None 
    
def probability_improvement(x, mu, S, beta, t_best, epsilon, **kwargs):
    return None
    
def lower_confidence_bound(x, mu, S, beta, t_best, gamma, **kwargs):
    return None

### ===========================

ALL_FUNCTIONS = [
    expected_improvement,
    probability_improvement,
    lower_confidence_bound,
]

(5pt) **2.a.2** Use the below code to plot the acqusition functions. Create one example that shows exploration and one example that shows exploitation. Comment on the observed behavior. 

In [ ]:
### ============= 2.a.2 Fill in below ==============
### Choose below values to encourage exploration vs. exploitation

# solution: 
kwargs_exploration = dict(
    epsilon = None,
    gamma = None,
    title = "More exploration" 
)
kwargs_exploitation = dict(
    epsilon = None,
    gamma = None,
    title = "More exploitation"
)
### ===========================

mu = m_0
S = S_0
data = []

x_new, t_new = generate_data(w=w_true, beta=beta, x=0.7)
x_new = float(np.squeeze(x_new))
t_best = t_new
x_best = x_new

# update posterior using the new data.
for x_new in [0.4, 0.6]:
    mu, S = analytical_posterior(mu, S, x_new, t_new, beta=beta)
    if t_best is None or t_new < t_best:
        t_best = t_new
        x_best = x_new

for kwargs in [kwargs_exploration, kwargs_exploitation]:
    print(kwargs.get("title"))
    fig, axs = plt.subplots(2, 1)
    fig.set_size_inches(10, 5)
    plot_function_distribution(axs[0], x_grid, t_true, data, mu, S)
    axs[0].set_title(kwargs.get("title"))
    axs[0].axhline(t_best, color="C0", ls=":")
    axs[0].axvline(x_best, color="C0", ls=":")
    plot_acquisition_functions_optimization(axs[1], x_grid, mu, S, beta, t_best, kwargs)

(10 pts) **2.a.3**  Run the below code using the three different acquisition functions.
- Do you find the optimal solution?
- How many steps does it take?
- How costly is each iteration?

For each acquisition function, reflect on how to change the epsilon/gamma parameter so that the optimum is found and report the results.

**Answer:** 

In [ ]:
kwargs = dict(
    epsilon = 1e-2,
    gamma = 1,
)

gamma_growth = 1.0

chosen_function = expected_improvement
# chosen_function = probability_improvement
# chosen_function = lower_confidence_bound

n_steps = 10  # number of inference steps
mu = m_0
S = S_0
data = []

# fixing the seed so that all methods start with the same points
np.random.seed(1)

x_new, t_new = generate_data(w=w_true, beta=beta)
x_new = float(np.squeeze(x_new))
t_best = t_new
x_best = x_new
for i in range(n_steps):
    # update posterior using the new data.
    mu, S = analytical_posterior(mu, S, x_new, t_new, beta=beta)

    kwargs["gamma"] *= gamma_growth

    # plotting
    data.append((x_new, t_new))
    fig, axes = plt.subplots(1, 2)
    plot_function_distribution(axes[0], x_grid, t_true, data, mu, S)
    plot_predictive_distribution(axes[1], x_grid, beta, K, mu, S)
    axes[1].set_ylim(-1, 1)
    plt.show()

    # doing negative below because we minimize by default.
    acquisition_grid = np.array(
        [chosen_function(np.array([x]), mu, S, beta, t_best, **kwargs) for x in x_grid]
    )
    x_opt = float(x_grid[np.argmax(acquisition_grid)])

    x_new, t_new = generate_data(x=x_opt, w=w_true, beta=beta)
    x_new = float(np.squeeze(x_new))
    if t_new <= t_best:
        t_best = t_new
        x_best = x_new

print(f"Best observed point: x = {x_best:.3f}, y = {t_best:.3f}")


## 2.b CMA-ES 

Next, you will use the off-the-shelve CMA-ES solver to solve the above toy problem. Make sure that you have installed the pycma package, by following the installation instructions given on Canvas. 

(10pt) **2.b.1** Tune the algorithm below until it converges. Describe what you changed from the defaults to get good convergence. 

**Answer:** 

In [ ]:
import cma
np.random.seed(1)

def run_cma_es(objective, m0, sigma0, popsize=6, max_evals=100, bounds=None):
    verb_disp = 0
    opts = {
        "popsize": popsize,
        "seed": 1,
        "verb_disp": verb_disp,
        "bounds": bounds
    }
    es = cma.CMAEvolutionStrategy(m0, sigma0, opts)

    cma_history = []
    while (es.result.evaluations < max_evals):
        X = es.ask()
        fit = [objective(x) for x in X]
        es.tell(X, fit)
        
        cma_history.append(
            {
                "iteration": i + 1,
                "evaluations": es.result.evaluations,
                "x": np.array(X),
                "y": np.array(fit),
                "mean": es.mean,
                "best_x": es.result.xbest,
                "best_y": es.result.fbest,
            }
        )
    return cma_history


objective = lambda x: generate_data(x=x.flatten()[0], w=w_true, beta=beta)[1]
d = 1
m0 = [0.5] * d
max_evals = 200

### ========== Fill in below ===============
### Tune the algorithm until the global minimum is found.
### You may only change below (and not, for eample, the initialization.
cma_history = run_cma_es(
    objective, 
    m0, 
    sigma0=0.5, 
    popsize=4,
    max_evals=max_evals
)
### ================================================

for step in cma_history:
    fig, ax = plt.subplots()
    ax.plot(x_grid, t_true, color="C0", label="Ground Truth")
    ax.scatter(step["x"], step["y"], color="C1", s=60, label="Samples")
    ax.axvline(step["mean"], color="C2", linestyle="--", label="Search mean")
    ax.scatter(
        step["best_x"],
        step["best_y"],
        color="C3",
        marker="*",
        s=10,
        label="Best so far",
    )
    ax.set_xlim(-0.5, 1)
    ax.set_ylim(-1, 1)
    ax.set_title(f"CMA-ES iteration {step['iteration']}")
    ax.legend(loc="upper left")
    plt.show()
### ================================

(5pt) **2.b.2** Now we design a method to invstigate performance. We propose to use the best observed objective value as a function of cumulative function evaluations. Plot the metric using the code below and discuss performance.

**Answer:** 

In [ ]:
# solution: 
eval_history_cmaes = [dict_["evaluations"] for dict_ in cma_history]
best_history_cmaes = [dict_["best_y"] for dict_ in cma_history]

plt.figure()
plt.plot(eval_history_cmaes, best_history_cmaes)
plt.xlabel("number of function evaluations")
plt.ylabel("objective")
plt.grid()
plt.title("CMA-ES performance")

## 2.c Performance Comparison

Finaly, you will run Bayesian Optimization, using an off-the-shelve implementation, and compare it with CMA-ES.

We will use the common global benchmark problems to do this. We choose the Rosenbrock function, shown below. 

(3pt) **2.c.1** What makes this function hard to optimize? 

**Answer:**

In [ ]:
from scipy.optimize import rosen as rosen_objective

def plot_rosenbrock_contour(ax, points=None, title="", bounds=(-2.0, 2.0)):
    x1 = np.linspace(bounds[0], bounds[1], 250)
    x2 = np.linspace(bounds[0], bounds[1], 250)
    X1, X2 = np.meshgrid(x1, x2)
    X = np.vstack([X1.flatten(), X2.flatten()])

    # Below does the same as
    # Z = 100.0 * (X2 - X1 ** 2) ** 2 + (1.0 - X1) ** 2
    Z = rosen_objective(np.array(X)).reshape((X1.shape))

    ax.contourf(X1, X2, np.log10(Z + 1e-3), levels=40)
    ax.scatter([1.0], [1.0], marker="*", s=140, color="red", label="optimum")
    ax.set_xlabel("x1")
    ax.set_ylabel("x2")
    ax.set_title(title)
    ax.legend(loc="upper left")
    
    if points is None:
        return
        
    points = np.asarray(points, dtype=float)
    if points.size > 0:
        ax.plot(points[:, 0], points[:, 1], "w.-", lw=1.5, ms=6, alpha=0.9)
        ax.scatter(points[0, 0], points[0, 1], color="k", s=50, label="start")

fig, ax = plt.subplots()
plot_rosenbrock_contour(ax)

We now use Bayesian optimization to optimize the same benchmark. First, as a sanity check, use the BO implementation of scikit-optimize, focusing on the same three acquisition functions that we implemented above. We provide hte interface to the off-the-shelve solver below. 

(5pt) **2.c.1** Run the below function and ensure that the results make sense by creating two plots: plotting the x,t pairs that are visited by each, and plotting the performance measure introduced above. Explain why the result of your handwritten BO and the off-the-shelve solver is not exactly the same.  

**Answer:** 

In [ ]:
from skopt import gp_minimize

bounds_2d = [(-0.5, 2.0)]
budget_2d = 60

res_dict = {}
for acq in ["EI", "PI", "LCB"]:
    print(f"Solving with {acq} (this may take a while...)")
    res_dict[acq] = gp_minimize(
        lambda x: generate_data(w_true, beta=beta, x=np.asarray(x, dtype=float))[1],
        dimensions=bounds_2d,
        acq_func=acq,
        n_calls=budget_2d,
        n_initial_points=6,
        random_state=1,
    )

In [ ]:
fig_fun, ax_fun = plt.subplots()
fig_cost, ax_cost = plt.subplots()
for acq, res in res_dict.items():
    ax_fun.scatter(res["x_iters"], res["func_vals"], label=acq)
    
    fun_best = np.minimum.accumulate(res["func_vals"])
    ax_cost.plot(np.arange(len(fun_best)), fun_best)
    ax_cost.set_xlabel("number of function evaluations")
    ax_cost.set_ylabel("objective")
    ax_cost.grid()


(20pt) **2.c.2** Now run both Bayesian Optimization and CMA-ES on the benchmarks for dimension 2. Complete the below code, and then comment on the performance. For the performance, check qualitatively the distribution of points sampled, over time (in 2D cost contour plots), and check the performance. What do you observe? Which method is preferable?

**Answer**: 

In [ ]:
bounds_2d = [(-2.0, 2.0), (-2.0, 2.0)]
budget_2d = 40


bo_rosen_2d = {}
for acq in ["EI", "PI", "LCB"]:
    print(f"Solving with {acq} (this may take a while...)")

    ### ======== Fill in below ===============
    ### Minimize the Rosenbrock function using BO and CMA-ES

    bo_rosen_2d[acq] = None
    
    cma_rosen_2d = None
    ### ================================

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for ax, (acq, res) in zip(axes, bo_results_2d.items()):
    plot_rosenbrock_contour(ax, points=np.asarray(res.x_iters, dtype=float), title=f"BO trajectory ({acq})")

mean_history = [dict_["mean"] for dict_ in cma_rosen_2d]
plot_rosenbrock_contour(axes[3], np.array(mean_history), "CMA-ES mean trajectory")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots()
for acq in ["EI", "PI", "LCB"]:
    ax.loglog(
    np.arange(1, len(bo_results_2d[acq].func_vals) + 1),
    np.minimum.accumulate(bo_results_2d[acq].func_vals),
    marker="o",
    label=f"BO ({acq})",
    )

eval_history = [dict_["evaluations"] for dict_ in cma_rosen_2d]
best_history = [dict_["best_y"] for dict_ in cma_rosen_2d]
ax.loglog(
    eval_history,
    best_history,
    marker="o",
    label="CMA-ES",
)

ax.set_xlabel("Function evaluations")
ax.set_ylabel("Best objective so far")
ax.set_title("2D Rosenbrock convergence")
ax.legend()
plt.tight_layout()
plt.show()

(10pt) **2.c.3** Repeat the analysis from point 2.c.2 but in high dimensions: d=10. What do you observe in terms of speed and performance? 

**Answer:** 

In [ ]:
d = 10
bounds_10d = [(-2.0, 2.0)] * d

### =========== Fill in below ==============
# Take above's code and increase the dimensions.
# Plot the results and comment. 

# solution: 
# TO BE IMPLEMENTED
### =================================

## Acknowledgment of Collaboration and/or Tool Use

Please choose from below (simply delete the lines that do not apply) and add a few additional notes

- “I worked alone on this assignment.”, or
- “I worked with ~~~~~~ [person or tool] on this assignment.” and/or
- “I received assistance from ~~~~~~ [person or tool] on this assignment.”

For the last two cases, specify how the person or tool helped you and explain why this amplified your learning process:

_add answer here_